### Prerequisitions

In [ ]:
!pip install mediapipe --user

In [ ]:
!pip install opencv-python

In [1]:
import math
import time
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

## Initialization

### Parameters

In [16]:
# Camera Settings
camera_index = 700
resolution = [640, 480]

# Model Settings
model_path = '../models/hand_landmarker.task'
no_of_hands = 1
hand_confidence = 0.7
tracking_confidence = 0.6

# Gesture Settings
pinch_dist_max = 35
click_dist_max = 35
min_finger_bend = 0.02

### Mediapipe HandLandmarker

In [17]:
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=no_of_hands,
    min_hand_detection_confidence=hand_confidence,
    min_tracking_confidence=tracking_confidence
)

In [18]:
detector = vision.HandLandmarker.create_from_options(options)

### OpenCV

In [19]:
cap = cv2.VideoCapture(camera_index)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, resolution[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, resolution[1])

True

## Gesture Test

In [20]:
def process_gestures(frame, result, w, h):
    status_text = "Tracking: Idle"
    click_text = "Click: Idle"

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        # Landmark 4 = Thumb Tip | Landmark 8 = Index Tip | Landmark 12 = Middle Tip
        thumb_tip = hand[4]
        idx_tip = hand[8]
        mid_tip = hand[12]

        # Convert to pixel coordinates
        thumb_x, thumb_y = int(thumb_tip.x * w), int(thumb_tip.y * h)
        idx_x, idx_y = int(idx_tip.x * w), int(idx_tip.y * h)
        mid_x, mid_y = int(mid_tip.x * w), int(mid_tip.y * h)

        # Distances
        pinch_dist = math.hypot(thumb_x - idx_x, thumb_y - idx_y)  # Thumb <-> Index
        click_dist = math.hypot(idx_x - mid_x, idx_y - mid_y)      # Index <-> Middle

        # Draw all 21 joints
        for lm in hand:
            cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 3, (0, 0, 255), cv2.FILLED)

        ### --- TRIGGERS --- ###
        
        # TRIGGER 1: Thumb + Index touching -> Tracking Active
        if pinch_dist < pinch_dist_max:
            status_text = f"TRACKING ACTIVE ({int(pinch_dist)}px)"
            # Green line between Thumb and Index
            cv2.line(frame, (thumb_x, thumb_y), (idx_x, idx_y), (0, 255, 0), 3)
            cv2.circle(frame, (idx_x, idx_y), 8, (0, 255, 0), cv2.FILLED)
            cv2.circle(frame, (thumb_x, thumb_y), 8, (0, 255, 0), cv2.FILLED)

            # TRIGGER 2: Middle touches Index while tracking is active -> Left Click
            if click_dist < click_dist_max:
                click_text = f"LEFT CLICK TRIGGERED! ({int(click_dist)}px)"
                # Cyan line between Index and Middle
                cv2.line(frame, (idx_x, idx_y), (mid_x, mid_y), (255, 255, 0), 3)
                cv2.circle(frame, (mid_x, mid_y), 10, (255, 255, 0), cv2.FILLED)
            else:
                cv2.circle(frame, (mid_x, mid_y), 6, (0, 255, 255), cv2.FILLED)
        else:
            # Idle visuals (Fingers apart)
            cv2.circle(frame, (idx_x, idx_y), 8, (0, 0, 255), cv2.FILLED)
            cv2.circle(frame, (thumb_x, thumb_y), 8, (255, 0, 255), cv2.FILLED)
            cv2.circle(frame, (mid_x, mid_y), 6, (180, 180, 180), cv2.FILLED)

    return frame, status_text, click_text

In [21]:
print("Running! Click the OpenCV window and press 'q' to stop.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)  # Mirrors the frame
    h, w, _ = frame.shape

    # MediaPipe inference
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB, 
        data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )
    timestamp_ms = int(time.time() * 1000)
    result = detector.detect_for_video(mp_image, timestamp_ms)

    # Process gestures using our function from Cell 2
    frame, status_text, click_text = process_gestures(frame, result, w, h)

    # HUD Overlay
    is_active = "TRACKING ACTIVE" in status_text
    cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, 0.75, 
                (0, 255, 0) if is_active else (0, 0, 255), 2)
    cv2.putText(frame, click_text, (20, 80), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, 0.75, 
                (0, 255, 255) if "TRIGGERED" in click_text else (180, 180, 180), 2)

    cv2.imshow("Hand Gesture Detection Tester", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
detector.close()
cv2.destroyAllWindows()

Running! Click the OpenCV window and press 'q' to stop.
